In [1]:
import torch

# Check if CUDA is available
if torch.cuda.is_available():
    # Get the number of available GPUs
    num_gpus = torch.cuda.device_count()
    print(f"Number of available GPUs: {num_gpus}")

    # Optional: print the name of each GPU
    for i in range(num_gpus):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("CUDA is not available. No GPUs found.")

Number of available GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [2]:
! pip install bitsandbytes
! pip install peft
! pip install --pre deepchem
! pip install ai2-olmo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.7 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 85.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 51.6 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you 

In [3]:
! pip install transformers==4.57.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 15.5 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.


In [4]:
! pip list | grep "transformers"

sentence-transformers                    5.2.3
transformers                             4.57.1


In [7]:
%%writefile train.py
import torch
import pytorch_lightning as pl
import deepchem as dc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType
)
from pytorch_lightning.callbacks import ModelCheckpoint
from deepchem.molnet import load_freesolv
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error
import seaborn as sns
import re
from torch.nn.functional import softmax
from pytorch_lightning.callbacks import EarlyStopping


class OlmoDataset(Dataset):
    def __init__(self, mode="Train", max_length=350):
        self.mode = mode.lower()

        self.tokenizer = AutoTokenizer.from_pretrained(
            "Codemaster67/ChemOlmo-7b",
            trust_remote_code=True,
        )
        self.tokenizer.padding_side = "right" if self.mode ==  "train" else "left"
        self.tokenizer.pad_token = self.tokenizer.eos_token

        tasks, datasets, transformers = load_freesolv(featurizer="raw", splitter='scaffold')
        train, valid, test = datasets

        self.task_names = tasks

        if self.mode == "train":
            self.data = train
        elif self.mode == "valid":
            self.data = valid
        elif self.mode == "test":
            self.data = test
        else:
            self.data = train

        self.max_length = max_length
        self.samples = []
        self._filldataset()

    def _filldataset(self):

        for i in range(len(self.data)):
            smiles = self.data.ids[i]
            labels = self.data.y[i]
            weights = self.data.w[i]  #it has some missing data too

            for task_idx, label in enumerate(labels):
                if weights[task_idx] > 0:
                    task_name = "hydration free energy"
                    self.samples.append(self._create_prompt(smiles, task_name, label))

        print(f"[{self.mode.upper()}] Number of samples: {len(self.samples)}")

    def _create_prompt(self, smiles, task_name,label):
        answer = f"{label:.5f}"

        full_prompt = (
            "### Instruction:\n"
            f"Predict the {task_name} (in kcal/mol) for the following molecule:\n{smiles}\n\n"
            "### Response:\n"
            f"{answer}"
        )
        return full_prompt

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        text = self.samples[idx] + self.tokenizer.eos_token # transformers version 4.57 does not add eos token unlike version 5.2
        encodings = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        input_ids = encodings["input_ids"].squeeze(0)
        attention_mask = encodings["attention_mask"].squeeze(0)
        labels = input_ids.clone()

        separator = "### Response:\n"
        parts = text.split(separator)

        if len(parts) >= 2:
            prompt_text = parts[0] + separator
            prompt_encodings = self.tokenizer(
                prompt_text,
                truncation=True,
                max_length=self.max_length,
                return_tensors="pt"
            )
            prompt_len = prompt_encodings["input_ids"].shape[1]

            if self.tokenizer.padding_side == "right":
                # Right-padded: [prompt | answer | EOS | PAD...]
                if prompt_len < len(labels):
                    labels[:prompt_len] = -100
            else:
                # Left-padded: [PAD... | prompt | answer | EOS]
                # Prompt starts after the padding tokens
                pad_len = (attention_mask == 0).sum().item()
                answer_start = pad_len + prompt_len
                if answer_start < len(labels):
                    labels[:answer_start] = -100

        # Mask padding positions using attention_mask (not token IDs)
        # This avoids accidentally masking the real EOS token at the end
        labels[attention_mask == 0] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

class OLMO_QLoRA(pl.LightningModule):
    def __init__(self):
        super().__init__()

        self.tokenizer = AutoTokenizer.from_pretrained(
            "Codemaster67/ChemOlmo-7b",
            trust_remote_code=True,
            padding_side="right"
        )
        self.tokenizer.pad_token = self.tokenizer.eos_token


        self.bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16
        )

        self.peft_config = LoraConfig(
            r=32,
            lora_alpha=64,
            target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
            lora_dropout=0.05,
            bias="none",
            task_type=TaskType.CAUSAL_LM #seq_cls or causal_lm for regression ?
        )
    def configure_model(self):
        self.model = AutoModelForCausalLM.from_pretrained(
            "Codemaster67/ChemOlmo-7b",
            quantization_config=self.bnb_config,
            trust_remote_code=True,
        )
        self.model = prepare_model_for_kbit_training(self.model)
        self.model = get_peft_model(self.model, self.peft_config)
        self.model.print_trainable_parameters()

    def forward(self, input_ids, attention_mask, labels=None):
        return self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
    def training_step(self, batch, batch_idx):
        outputs = self(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )
        loss = outputs.loss
        self.log("Train_loss", loss, prog_bar=True, on_step=True, on_epoch=True, logger=True)

        return loss

    def validation_step(self, batch, batch_idx):
        outputs = self(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )
        loss = outputs.loss
        self.log("val_loss", loss, prog_bar=True, sync_dist=True)
        return loss
    def on_train_end(self):
            if self.trainer.is_global_zero:
                print("\nStarting test set evaluation (RMSE) after training...")

                test_dataset = OlmoDataset(mode="test", max_length=350)
                test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

                self.model.eval()

                preds = []
                trues = []

                # Get ground truth values
                true_values = test_dataset.data.y.flatten()

                print(f"Evaluating on {len(test_loader)} samples...")

                with torch.no_grad():
                    for i, batch in enumerate(test_loader):
                        # Move batch to the correct device
                        batch = {k: v.to(self.device) for k, v in batch.items()}

                        input_ids = batch["input_ids"]
                        labels = batch["labels"]
                        attention_mask = batch["attention_mask"]

                        # Logic to slice the input so we only feed the prompt to generate()
                        # We look for the first index where labels are NOT -100 (which marks the start of the answer)
                        response_mask = (labels != -100)
                        if response_mask.sum() == 0:
                            print(f"Sample {i}: no answer tokens found, skipping")
                            continue

                        answer_start_index = response_mask.int().argmax(dim=1).item()

                        # Slice input_ids to keep only the instruction + input (remove the ground truth answer)
                        if answer_start_index > 0:
                            prompt_ids = input_ids[:, :answer_start_index]
                            prompt_mask = attention_mask[:, :answer_start_index]
                        else:
                            prompt_ids = input_ids
                            prompt_mask = attention_mask

                        if i % 10 == 0:
                            print(self.tokenizer.decode(prompt_ids[0],skip_special_tokens=True))

                        outputs = self.model.generate(
                            input_ids=prompt_ids,
                            attention_mask=prompt_mask,
                            max_new_tokens=10,
                            pad_token_id=self.tokenizer.eos_token_id,
                            do_sample=False
                        )

                        generated_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
                        # Extract the number from the response



                        try:
                            if "### Response:" in generated_text:
                                response_part = generated_text.split("### Response:")[-1].strip()
                            else:
                                response_part = generated_text.strip()

                            # Regex to find float numbers (handles negative and decimals)


                            if i % 10 == 0:
                                print(f"Sample {i}:")
                                print(self.tokenizer.decode(prompt_ids[0],skip_special_tokens=True))
                                print("-------------response-------------")
                                print(response_part)
                                print("-------------response-------------")

                            match = re.search(r"-?\d+(?:\.\d+)?", response_part)
                            if match:
                                val = float(match.group())
                            else:
                                print(f"Warning: Could not parse number from: {response_part[:50]}...")
                                val = 0.0
                        except Exception as e:
                            print(f"Error parsing prediction: {e}")
                            val = 0.0

                        preds.append(val)
                        trues.append(true_values[i])

                        if i % 10 == 0:
                            print(f"Sample {i}: True={true_values[i]:.5f}, Pred={val:.5f}")

                preds = np.array(preds)
                trues = np.array(trues)

                # Calculate RMSE
                rmse = np.sqrt(mean_squared_error(trues, preds))

                print("\n=== Test Set Metrics ===")
                print(f"RMSE: {rmse:.4f}")

                # Save the metric to the model object so it can be accessed in the main loop
                self.final_rmse = rmse


    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=1e-4,weight_decay=1e-4)

        total_steps = self.trainer.estimated_stepping_batches

        warmup_steps = int(0.10*total_steps)
        scheduler_warmup = LinearLR(
            optimizer,
            start_factor=0.01,
            end_factor=1.0,
            total_iters=warmup_steps,
        )


        scheduler_cosine = CosineAnnealingLR(
            optimizer,
            T_max=total_steps - warmup_steps,
        )

        scheduler = SequentialLR(
            optimizer,
            schedulers=[scheduler_warmup, scheduler_cosine],
            milestones=[warmup_steps]
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "step"
            }
        }


if __name__ == "__main__":
    num_runs = 3
    rmse_scores = []

    for i in range(num_runs):
        print(f"\n======================================")
        print(f"        STARTING RUN {i + 1} OF {num_runs}        ")
        print(f"======================================\n")

        # Change seed per run to ensure variation
        pl.seed_everything(42 + i, workers=True)

        train_dataset = OlmoDataset(mode="train")
        val_dataset = OlmoDataset(mode="valid")

        train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

        # Setup Early Stopping based on validation loss
        early_stop_callback = EarlyStopping(
            monitor="val_loss",
            min_delta=0.00,
            patience=3,
            verbose=True,
            mode="min"
        )

        trainer = pl.Trainer(
            accelerator="gpu",
            devices=2,
            max_epochs=16,
            strategy= "ddp",
            precision="16-mixed",
            accumulate_grad_batches=8,
            enable_checkpointing=False,
            enable_progress_bar=False,
            gradient_clip_val=1,
            callbacks=[early_stop_callback]
        )

        model = OLMO_QLoRA()

        # Pass both train and val loaders to fit() for EarlyStopping to work
        trainer.fit(model, train_loader, val_loader)

        # Extract the RMSE score for this run (only on the main process)
        if trainer.is_global_zero:
            rmse_scores.append(model.final_rmse)

    # Calculate and print Final Statistics (only on the main process)
    if torch.distributed.is_initialized():
        rank = torch.distributed.get_rank()
    else:
        rank = 0

    if rank == 0 and len(rmse_scores) > 0:
        mean_rmse = np.mean(rmse_scores)
        std_rmse = np.std(rmse_scores)

        print("\n\n######################################")
        print("         FINAL TRAINING SUMMARY       ")
        print("######################################")
        print(f"Number of runs evaluated: {len(rmse_scores)}")
        print(f"RMSE Scores: {rmse_scores}")
        print(f"Mean RMSE: {mean_rmse:.4f}")
        print(f"Std RMSE:  {std_rmse:.4f}")
        print("######################################\n")


Overwriting train.py


In [8]:
! python train.py

No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
2026-05-23 18:31:13.637541: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779561073.885965     148 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779561073.960439     148 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin